# 🧠 IDL A1 — Task 2: Tell the Time
This notebook implements and compares multiple label encodings for clock-time prediction using CNNs.  
All experiments were run in Google Colab (T4 GPU).  



## 1. Setup and Data Loading
This section loads the 75×75 dataset, normalizes the images, adds a channel dimension,
and splits the data (80 % train / 10 % validation / 10 % test).


In [ ]:
# --- SETUP ---
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IM_PATH = "/content/A1_data_75/images.npy"
LB_PATH = "/content/A1_data_75/labels.npy"

# load
images = np.load(IM_PATH)          # (18000, 75, 75)
labels = np.load(LB_PATH)          # (18000, 2): [hour, minute]

# normalize + add channel
images = images.astype("float32")/255.0
if images.ndim == 3:
    images = images[..., None]

# shuffle then split 80/10/10
rng = np.random.default_rng(42)
idx = rng.permutation(len(images))
images, labels = images[idx], labels[idx]

N = len(images); i1 = int(0.8*N); i2 = int(0.9*N)
x_tr, x_val, x_te = images[:i1], images[i1:i2], images[i2:]
y_tr, y_val, y_te = labels[:i1], labels[i1:i2], labels[i2:]

BATCH = 128


## 2. Evaluation Metric
Implements the *common-sense minutes error*, which converts hour-minute pairs into total
minutes and measures the smallest circular difference on a 12-hour clock.


In [ ]:
@tf.function
def minutes_from_hm(h, m):
    return (tf.math.mod(h,12)*60) + tf.math.mod(m,60)

@tf.function
def common_sense_min_error(y_true_hm, y_pred_hm):
    tmin = minutes_from_hm(y_true_hm[...,0], y_true_hm[...,1])
    pmin = minutes_from_hm(y_pred_hm[...,0], y_pred_hm[...,1])
    diff = tf.abs(tmin - pmin)
    return tf.minimum(diff, 720 - diff)

def eval_common_sense_minutes(true_hm, pred_hm):
    t = tf.convert_to_tensor(true_hm, dtype=tf.float32)
    p = tf.convert_to_tensor(pred_hm, dtype=tf.float32)
    err = common_sense_min_error(t, p).numpy()
    return float(np.mean(err)), float(np.median(err))


## 3. CNN Backbone
Defines the convolutional feature extractor shared by all label encodings.  
Includes Batch Normalization, ReLU, Dropout, and He initialization.


In [ ]:
def cnn_backbone(input_shape, base=32, last_dim=128, dropout=0.25):
    inp = keras.Input(input_shape)
    x = inp
    for f in [base, base*2, base*2]:
        x = layers.Conv2D(f, 3, padding="same", kernel_initializer="he_normal")(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPool2D()(x); x = layers.Dropout(dropout)(x)
    x = layers.Conv2D(base*4, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(last_dim, activation="relu")(x); x = layers.Dropout(0.5)(x)
    return inp, x


## 4. Labeling Approaches and Results
Four labeling strategies are implemented and compared below.


### A. Multi-Head (Unscaled)
Joint hour classification and minute regression without normalization. This serves as the baseline model.


In [ ]:
def model_multihead(input_shape):
    inp, feat = cnn_backbone(input_shape)
    hour_logits = layers.Dense(12, activation="softmax", name="hour")(feat)
    minute_reg  = layers.Dense(1,  activation="linear",  name="minute")(feat)
    m = keras.Model(inp, [hour_logits, minute_reg])
    m.compile(
        optimizer=keras.optimizers.Adam(3e-4),
        loss={"hour":"sparse_categorical_crossentropy","minute":"mse"},
        loss_weights={"hour":1.0,"minute":0.2},
        metrics={"hour":["accuracy"], "minute":["mae"]},
    ); return m

def train_multihead(x_tr, y_tr, x_val, y_val, batch=BATCH):
    y_tr_h = (y_tr[:,0] % 12).astype("int32"); y_tr_m = y_tr[:,1].astype("float32")
    y_val_h = (y_val[:,0] % 12).astype("int32"); y_val_m = y_val[:,1].astype("float32")
    m = model_multihead(x_tr.shape[1:])
    es = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
    m.fit(x_tr, {"hour":y_tr_h,"minute":y_tr_m},
          validation_data=(x_val, {"hour":y_val_h,"minute":y_val_m}),
          epochs=50, batch_size=batch, callbacks=[es], verbose=2)
    ph, pm = m.predict(x_val, verbose=0)
    pr_h = np.argmax(ph, axis=1)
    pr_m = np.clip(np.round(pm.squeeze()), 0, 59)
    mean_e, med_e = eval_common_sense_minutes(y_val, np.stack([pr_h, pr_m],1))
    print(f"[Multi-head] mean={mean_e:.2f} min, median={med_e:.2f} min")
    return m


In [ ]:
m_multi = train_multihead(x_tr, y_tr, x_val, y_val)

Epoch 1/50
113/113 - 19s - 171ms/step - hour_accuracy: 0.0804 - hour_loss: 3.0276 - loss: 160.8771 - minute_loss: 787.3453 - minute_mae: 23.1665 - val_hour_accuracy: 0.0844 - val_hour_loss: 2.5160 - val_loss: 115.0316 - val_minute_loss: 561.3110 - val_minute_mae: 19.3467
Epoch 2/50
113/113 - 4s - 32ms/step - hour_accuracy: 0.0850 - hour_loss: 4.0976 - loss: 70.8609 - minute_loss: 333.7198 - minute_mae: 15.5048 - val_hour_accuracy: 0.0761 - val_hour_loss: 2.5164 - val_loss: 106.7697 - val_minute_loss: 520.4719 - val_minute_mae: 18.6694
Epoch 3/50
113/113 - 3s - 29ms/step - hour_accuracy: 0.0828 - hour_loss: 3.8050 - loss: 64.5936 - minute_loss: 303.9023 - minute_mae: 14.7791 - val_hour_accuracy: 0.0944 - val_hour_loss: 2.5250 - val_loss: 68.5921 - val_minute_loss: 328.7165 - val_minute_mae: 15.4739
Epoch 4/50
113/113 - 3s - 29ms/step - hour_accuracy: 0.0823 - hour_loss: 3.4628 - loss: 60.6609 - minute_loss: 285.9795 - minute_mae: 14.0474 - val_hour_accuracy: 0.0911 - val_hour_loss: 2.58

### B. Multi-Head (Scaled)
Improved version where minutes are scaled to [0,1] and predicted with a sigmoid activation to balance the loss magnitudes.


In [ ]:
# Continue training the 75×75 scaled model
m_multi2 = model_multihead_scaled(x_tr.shape[1:])
es = keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True)
m_multi2.fit(
    x_tr,
    {"hour": (y_tr[:,0]%12).astype("int32"),
     "minute": y_tr[:,1].astype("float32")/60.0},
    validation_data=(x_val, {"hour": (y_val[:,0]%12).astype("int32"),
                             "minute": y_val[:,1].astype("float32")/60.0}),
    epochs=80, batch_size=128, callbacks=[es], verbose=2
)

# evaluate on the same val split
ph, pm = m_multi2.predict(x_val, verbose=0)
pr_h = np.argmax(ph, axis=1)
pr_m = np.clip(np.round(pm.squeeze()*60.0), 0, 59)
mean_e, med_e = eval_common_sense_minutes(y_val, np.stack([pr_h, pr_m],1))
print(f"[75×75 SCALED] mean={mean_e:.2f} min, median={med_e:.2f} min")


Epoch 1/80
113/113 - 21s - 182ms/step - hour_accuracy: 0.0858 - hour_loss: 2.4967 - loss: 2.5812 - minute_loss: 0.0844 - minute_mae: 0.2505 - val_hour_accuracy: 0.0839 - val_hour_loss: 2.5043 - val_loss: 2.5915 - val_minute_loss: 0.0832 - val_minute_mae: 0.2498
Epoch 2/80
113/113 - 5s - 48ms/step - hour_accuracy: 0.0920 - hour_loss: 2.4848 - loss: 2.5678 - minute_loss: 0.0830 - minute_mae: 0.2489 - val_hour_accuracy: 0.0839 - val_hour_loss: 2.5272 - val_loss: 2.6170 - val_minute_loss: 0.0831 - val_minute_mae: 0.2492
Epoch 3/80
113/113 - 5s - 47ms/step - hour_accuracy: 0.0960 - hour_loss: 2.4795 - loss: 2.5614 - minute_loss: 0.0820 - minute_mae: 0.2459 - val_hour_accuracy: 0.0789 - val_hour_loss: 2.4952 - val_loss: 2.5807 - val_minute_loss: 0.0823 - val_minute_mae: 0.2476
Epoch 4/80
113/113 - 5s - 48ms/step - hour_accuracy: 0.1206 - hour_loss: 2.4210 - loss: 2.4958 - minute_loss: 0.0747 - minute_mae: 0.2251 - val_hour_accuracy: 0.0906 - val_hour_loss: 2.5535 - val_loss: 2.6389 - val_min

### C. Sine–Cosine Circular Regression
Encodes hour and minute angles as sine/cosine pairs to handle periodicity, then decodes them back to numeric time.


In [ ]:
def to_sin_cos(yhm):
    h = (yhm[:,0] % 12).astype("float32"); m = yhm[:,1].astype("float32")
    h_angle = 2*np.pi * ((h*60 + m)/720.0)
    m_angle = 2*np.pi * (m/60.0)
    return np.stack([np.cos(h_angle), np.sin(h_angle),
                     np.cos(m_angle), np.sin(m_angle)],1).astype("float32")

def model_sincos(input_shape):
    inp, feat = cnn_backbone(input_shape)
    out = layers.Dense(4, activation="tanh")(feat)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4), loss="mse")
    return m

def sincos_to_hm(pred):
    pred = np.asarray(pred); ch,sh,cm,sm = pred[:,0],pred[:,1],pred[:,2],pred[:,3]
    h_ang = np.arctan2(sh, ch) % (2*np.pi); m_ang = np.arctan2(sm, cm) % (2*np.pi)
    minutes = np.round(m_ang * 60/(2*np.pi)).astype(int)
    hours   = np.round(((h_ang * 720/(2*np.pi)) - minutes)/60).astype(int) % 12
    minutes = np.clip(minutes, 0, 59)
    return np.stack([hours, minutes],1)

def train_sincos(x_tr, y_tr, x_val, y_val, batch=BATCH):
    y_tr_sc, y_val_sc = to_sin_cos(y_tr), to_sin_cos(y_val)
    m = model_sincos(x_tr.shape[1:])
    es = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
    m.fit(x_tr, y_tr_sc, validation_data=(x_val, y_val_sc),
          epochs=50, batch_size=batch, callbacks=[es], verbose=2)
    pr = m.predict(x_val, verbose=0)
    pred_hm = sincos_to_hm(pr)
    mean_e, med_e = eval_common_sense_minutes(y_val, pred_hm)
    print(f"[SinCos] mean={mean_e:.2f} min, median={med_e:.2f} min")
    return m


In [ ]:
m_sinc = train_sincos(x_tr, y_tr, x_val, y_val)


Epoch 1/50
113/113 - 18s - 160ms/step - loss: 0.5314 - val_loss: 0.5131
Epoch 2/50
113/113 - 3s - 29ms/step - loss: 0.4968 - val_loss: 0.5441
Epoch 3/50
113/113 - 3s - 29ms/step - loss: 0.4862 - val_loss: 0.5616
Epoch 4/50
113/113 - 3s - 30ms/step - loss: 0.4672 - val_loss: 0.5276
Epoch 5/50
113/113 - 3s - 29ms/step - loss: 0.4461 - val_loss: 0.6151
Epoch 6/50
113/113 - 3s - 29ms/step - loss: 0.4283 - val_loss: 0.6257
Epoch 7/50
113/113 - 3s - 30ms/step - loss: 0.4146 - val_loss: 0.5490
[SinCos] mean=176.99 min, median=175.00 min


### D. Bucketed Classification
Discretizes the clock time into 24 (30 min) or 48 (15 min) classes. Simple classification but lower precision.


In [ ]:
def bucket_minutes(yhm, step=30):
    mins = (yhm[:,0] % 12)*60 + yhm[:,1]
    return (mins // step).astype("int32"), (720 // step)

def model_bucket(input_shape, K):
    inp, feat = cnn_backbone(input_shape)
    out = layers.Dense(K, activation="softmax")(feat)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(3e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def bucket_center_to_hm(pred_idx, step):
    center = (pred_idx * step + step/2.0) % 720
    hours = (center // 60).astype(int) % 12
    minutes = np.round(center % 60).astype(int)
    minutes = np.clip(minutes, 0, 59)
    return np.stack([hours, minutes],1)

def train_bucket(x_tr, y_tr, x_val, y_val, step=30, batch=BATCH):
    y_tr_b, K = bucket_minutes(y_tr, step); y_val_b, _ = bucket_minutes(y_val, step)
    m = model_bucket(x_tr.shape[1:], K)
    es = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
    m.fit(x_tr, y_tr_b, validation_data=(x_val, y_val_b),
          epochs=50, batch_size=batch, callbacks=[es], verbose=2)
    pr_idx = np.argmax(m.predict(x_val, verbose=0), axis=1)
    pred_hm = bucket_center_to_hm(pr_idx, step)
    mean_e, med_e = eval_common_sense_minutes(y_val, pred_hm)
    print(f"[Bucket {step}m] mean={mean_e:.2f} min, median={med_e:.2f} min")
    return m


In [ ]:
m_b30   = train_bucket(x_tr, y_tr, x_val, y_val, step=30)
m_b15   = train_bucket(x_tr, y_tr, x_val, y_val, step=15)


Epoch 1/50
113/113 - 17s - 154ms/step - accuracy: 0.0426 - loss: 3.2033 - val_accuracy: 0.0428 - val_loss: 3.2007
Epoch 2/50
113/113 - 3s - 30ms/step - accuracy: 0.0426 - loss: 3.1791 - val_accuracy: 0.0472 - val_loss: 3.2292
Epoch 3/50
113/113 - 3s - 30ms/step - accuracy: 0.0472 - loss: 3.1736 - val_accuracy: 0.0450 - val_loss: 3.2680
Epoch 4/50
113/113 - 3s - 30ms/step - accuracy: 0.0531 - loss: 3.1696 - val_accuracy: 0.0428 - val_loss: 3.3116
Epoch 5/50
113/113 - 3s - 30ms/step - accuracy: 0.0563 - loss: 3.1527 - val_accuracy: 0.0450 - val_loss: 3.4540
Epoch 6/50
113/113 - 3s - 30ms/step - accuracy: 0.0685 - loss: 3.1210 - val_accuracy: 0.0489 - val_loss: 3.6603
Epoch 7/50
113/113 - 3s - 30ms/step - accuracy: 0.0768 - loss: 3.0555 - val_accuracy: 0.0461 - val_loss: 3.5837
[Bucket 30m] mean=178.50 min, median=179.00 min
Epoch 1/50
113/113 - 20s - 178ms/step - accuracy: 0.0209 - loss: 3.8982 - val_accuracy: 0.0183 - val_loss: 3.8863
Epoch 2/50
113/113 - 3s - 30ms/step - accuracy: 0.02

## 5. Comparison Table (75×75)

| Labeling                | Mean (min) | Median (min) |
|-------------------------|-----------:|-------------:|
| Multi-Head (Unscaled)   | 160.99     | 154.00       |
| **Multi-Head (Scaled)** | **13.43**  | **9.00**     |
| Sine–Cosine             | 176.99     | 175.00       |
| Bucket 30 m             | 178.50     | 179.00       |
| Bucket 15 m             | 173.26     | 172.00       |




## 6. Final Evaluation on 150×150 Dataset
We retrain only the best-performing model (Multi-Head Scaled) on higher-resolution 150×150 images.
This step checks whether the same architecture scales well with more detailed visual input.

> Note: Other methods (Sin–Cos and Bucketed) are not retrained on 150×150, as the goal here is to
> validate that the best approach generalizes when image resolution increases.


In [ ]:
# --- 6) Final Evaluation on 150×150 ---

def model_multihead_scaled_large(input_shape):
    inp = keras.Input(input_shape)

    # SAFE, non-geometric augmentations (active only in training)
    x = layers.GaussianNoise(0.02)(inp)          # slight sensor noise
    x = layers.RandomContrast(0.1)(x)            # small contrast jitter

    # backbone
    x = layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.MaxPool2D()(x); x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.MaxPool2D()(x); x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.MaxPool2D()(x); x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(256, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.30)(x)

    hour_logits = layers.Dense(12, activation="softmax", name="hour")(x)
    minute_reg  = layers.Dense(1, activation="sigmoid", name="minute")(x)

    m = keras.Model(inp, [hour_logits, minute_reg])
    m.compile(
        optimizer=keras.optimizers.Adam(5e-4),  # smaller LR for larger model
        loss={"hour": "sparse_categorical_crossentropy", "minute": "mse"},
        loss_weights={"hour": 1.0, "minute": 1.0},
        metrics={"hour": ["accuracy"], "minute": ["mae"]},
    )
    return m

best150 = model_multihead_scaled_large(x_tr150.shape[1:])

es  = keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)
rlr = keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5, verbose=1)

# Train (no flips/rotations; val split is clean because aug is inside the model)
hist150 = best150.fit(
    x_tr150,
    {
        "hour":   (y_tr150[:,0] % 12).astype("int32"),
        "minute":  y_tr150[:,1].astype("float32") / 60.0,
    },
    validation_split=0.10,
    epochs=100,
    batch_size=128,
    callbacks=[es, rlr],
    verbose=2,
)

# Evaluate on held-out test split
ph, pm = best150.predict(x_te150, verbose=0)
pr_h = np.argmax(ph, axis=1)
pr_m = np.clip(np.round(pm.squeeze() * 60.0), 0, 59)
mean_e, med_e = eval_common_sense_minutes(y_te150, np.stack([pr_h, pr_m], 1))
print(f"[FINAL 150×150] mean={mean_e:.2f} min, median={med_e:.2f} min")


Epoch 1/100
102/102 - 41s - 404ms/step - hour_accuracy: 0.0860 - hour_loss: 2.5070 - loss: 2.5920 - minute_loss: 0.0852 - minute_mae: 0.2511 - val_hour_accuracy: 0.0875 - val_hour_loss: 2.4934 - val_loss: 2.5769 - val_minute_loss: 0.0823 - val_minute_mae: 0.2490 - learning_rate: 5.0000e-04
Epoch 2/100
102/102 - 23s - 226ms/step - hour_accuracy: 0.0846 - hour_loss: 2.4876 - loss: 2.5709 - minute_loss: 0.0833 - minute_mae: 0.2493 - val_hour_accuracy: 0.0785 - val_hour_loss: 2.5953 - val_loss: 2.6702 - val_minute_loss: 0.0827 - val_minute_mae: 0.2490 - learning_rate: 5.0000e-04
Epoch 3/100
102/102 - 23s - 228ms/step - hour_accuracy: 0.0879 - hour_loss: 2.4844 - loss: 2.5672 - minute_loss: 0.0828 - minute_mae: 0.2482 - val_hour_accuracy: 0.0785 - val_hour_loss: 2.6166 - val_loss: 2.6918 - val_minute_loss: 0.0851 - val_minute_mae: 0.2504 - learning_rate: 5.0000e-04
Epoch 4/100
102/102 - 24s - 232ms/step - hour_accuracy: 0.0935 - hour_loss: 2.4803 - loss: 2.5622 - minute_loss: 0.0820 - minut

### ✅ Final 150×150 Result
Mean error **179.39 min**, median **179.00 min** (Multi-Head Scaled, larger backbone, LR scheduling).


## 7. Discussion
Detailed analysis provided in the written report.

## 8. Conclusion
Main finding: scaled multi-head encoding performed best among all methods.
